In [1]:
pip install transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.0 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=ff2f9aa046c958c4d9c45c0f82c7d1818f2fdc5d67c625d18d771a55bde2d720
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [2]:
"""
Phase 2 -- first end-to-end run: running-total environment on GPT-2,
Endogeneity + Policy-Relevance via activation patching, routed through
KnowledgeUnitView / QuestionGenerator / EvidenceManager.

Ports from self-model-agent (verbatim where noted, per this project's
"reuse validated code, don't rewrite it" discipline):
  - evaluation/stage1_endogeneity.py: conditional_granger_f_test, benjamini_hochberg
  - core/hypothesis/hypothesis_engine.py: HypothesisEngine
  - core/world_model/world_model.py: NodeStatus, Node, WorldModel
  - core/self_model/self_graph.py: NodeKind, SelfGraphNode (Kind axis only;
    full edge-typed SelfGraph is not needed for this validation run)

New in this script (design: docs/phase2_design_consolidated.md,
docs/knowledge_unit_reconciliation.md):
  - KnowledgeUnitView, EvidenceManager, QuestionGenerator
  - RunningTotalEnvironment (the autoregressive loop Endogeneity needs)
  - candidate signal extraction (fixed, out-of-sample-calibrated probe
    directions) and a patching-based Policy-Relevance flip-rate test

STATUS: UNTESTED, written outside Colab. Expect debugging. Known
simplifications, flagged inline where they occur -- read these before
trusting any printed result:
  1. Prompt template deviates from the design doc's illustrative
     "Total: {T}. Add {n}. New total:" phrasing -- uses Phase 1's
     VALIDATED "{T} + {n} =" format instead, looped. Rationale: Phase 1
     never tested the naturalistic phrasing at all; reusing the
     validated one maximizes the chance the loop produces parseable,
     on-distribution generations. If this works, testing the more
     naturalistic phrasing is a reasonable follow-up, not a prerequisite.
  2. Action encoding is a plain one-hot over digits 0-9 (10 categories),
     not the k-1 reduced encoding evaluation/action_encoding.py uses for
     the non-LLM track. Simplification for this first run; the ported
     conditional_granger_f_test doesn't require the reduction, it's a
     convention from the original codebase, not a statistical necessity.
  3. Policy-Relevance ACCEPTED/REJECTED threshold on flip_rate (0.3) is
     PROVISIONAL -- not calibrated against anything in this run. Treat
     the raw flip_rate number as the real result; the accept/reject
     label is a first guess at where to draw the line.
  4. handle_id for LLM activations is an ad hoc string (per
     docs/phase2_design_consolidated.md §3.3's open item), not a real
     InterventionHandleRegistry entry -- fine for this validation run,
     not fine to scale past a couple of hand-picked candidates.
"""

import numpy as np
import torch
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional
from scipy import stats
from sklearn.linear_model import Ridge
from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
model.eval()
N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads

RNG = np.random.default_rng(0)

# =============================================================================
# 1. PORTED, VERBATIM: conditional Granger F-test + BH-FDR
#    (copied from evaluation/stage1_endogeneity.py -- do not modify the
#    statistical logic here; this is validated, tested code)
# =============================================================================

LAG_ORDER = 3
ALPHA = 0.05


def build_lagged(matrix: np.ndarray, lag: int) -> np.ndarray:
    T = matrix.shape[0]
    cols = [matrix[lag - k: T - k] for k in range(1, lag + 1)]
    return np.concatenate(cols, axis=1)


def ols_rss(y: np.ndarray, X: np.ndarray) -> float:
    X_design = np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)
    beta, _, _, _ = np.linalg.lstsq(X_design, y, rcond=None)
    resid = y - X_design @ beta
    return float(resid @ resid)


def conditional_granger_f_test(target_col: np.ndarray, other_cols: np.ndarray,
                                action_onehot: np.ndarray, lag: int = LAG_ORDER):
    T = target_col.shape[0]
    y = target_col[lag:]
    n = y.shape[0]

    self_lags = build_lagged(target_col.reshape(-1, 1), lag)
    other_lags = build_lagged(other_cols, lag) if other_cols.shape[1] > 0 else np.zeros((n, 0))

    action_contemporaneous = action_onehot[lag:]
    action_past_lags = build_lagged(action_onehot, lag)
    action_history = np.concatenate([action_contemporaneous, action_past_lags], axis=1)

    restricted_X = np.concatenate([self_lags, other_lags], axis=1)
    unrestricted_X = np.concatenate([restricted_X, action_history], axis=1)

    k_r = restricted_X.shape[1] + 1
    k_u = unrestricted_X.shape[1] + 1
    q = k_u - k_r

    rss_r = ols_rss(y, restricted_X)
    rss_u = ols_rss(y, unrestricted_X)

    df1 = q
    df2 = n - k_u
    if df2 <= 0 or rss_u <= 0:
        return float("nan"), 1.0

    f_stat = ((rss_r - rss_u) / df1) / (rss_u / df2)
    f_stat = max(f_stat, 0.0)
    p_value = stats.f.sf(f_stat, df1, df2)
    return f_stat, p_value


def benjamini_hochberg(p_values: list, alpha: float = ALPHA) -> list:
    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = np.array(p_values)[order]
    thresholds = (np.arange(1, m + 1) / m) * alpha
    passed = sorted_p <= thresholds
    if not passed.any():
        cutoff_rank = -1
    else:
        cutoff_rank = np.max(np.where(passed))
    reject = np.zeros(m, dtype=bool)
    if cutoff_rank >= 0:
        reject[order[: cutoff_rank + 1]] = True
    return reject.tolist()


# =============================================================================
# 2. PORTED, VERBATIM: HypothesisEngine
#    (copied from core/hypothesis/hypothesis_engine.py)
# =============================================================================

class HypothesisEngine:
    def __init__(self, signal_ids: list, lag: int = LAG_ORDER, alpha: float = ALPHA):
        self.signal_ids = signal_ids
        self.lag = lag
        self.alpha = alpha
        self._signal_rows: list = []
        self._action_rows: list = []

    def record_step(self, signal_values_by_id: dict, action_onehot_row: list):
        self._signal_rows.append([signal_values_by_id[sid] for sid in self.signal_ids])
        self._action_rows.append(action_onehot_row)

    def n_samples(self) -> int:
        return len(self._signal_rows)

    def test_endogeneity(self, min_samples: int = 60) -> dict:
        if self.n_samples() < min_samples:
            return {}
        matrix = np.array(self._signal_rows)
        action_onehot = np.array(self._action_rows)
        p_values = []
        for i in range(len(self.signal_ids)):
            target = matrix[:, i]
            others = np.delete(matrix, i, axis=1)
            _, p = conditional_granger_f_test(target, others, action_onehot, lag=self.lag)
            p_values.append(p)
        rejected = benjamini_hochberg(p_values, alpha=self.alpha)
        return {
            sid: {"p_value": p_values[i], "significant": rejected[i]}
            for i, sid in enumerate(self.signal_ids)
        }


# =============================================================================
# 3. PORTED, VERBATIM: WorldModel (NodeStatus, Node, accept/reject/add_candidate)
#    (copied from core/world_model/world_model.py)
# =============================================================================

class NodeStatus(Enum):
    CANDIDATE = "candidate"
    ACCEPTED = "accepted"
    REJECTED = "rejected"
    UNRESOLVED = "unresolved"


@dataclass
class Node:
    signal_id: str
    status: NodeStatus
    endogeneity_p_value: Optional[float] = None
    best_handle_id: Optional[str] = None
    flip_rate: Optional[float] = None
    entity_id: Optional[int] = None
    evidence: dict = field(default_factory=dict)


class WorldModel:
    def __init__(self):
        self.nodes: Dict[str, Node] = {}

    def add_candidate(self, signal_id: str, p_value: float):
        if signal_id not in self.nodes:
            self.nodes[signal_id] = Node(signal_id=signal_id, status=NodeStatus.CANDIDATE,
                                          endogeneity_p_value=p_value)

    def accept(self, signal_id: str, handle_id: str, flip_rate: float, evidence: Optional[dict] = None):
        node = self.nodes[signal_id]
        node.status = NodeStatus.ACCEPTED
        node.best_handle_id = handle_id
        node.flip_rate = flip_rate
        if evidence:
            node.evidence.update(evidence)

    def reject(self, signal_id: str, handle_id: Optional[str] = None, flip_rate: Optional[float] = None):
        node = self.nodes[signal_id]
        node.status = NodeStatus.REJECTED
        if handle_id:
            node.best_handle_id = handle_id
            node.flip_rate = flip_rate

    def self_model_summary(self) -> str:
        lines = []
        by_status: Dict[NodeStatus, List[str]] = {}
        for sid, node in self.nodes.items():
            by_status.setdefault(node.status, []).append(sid)
        for status in NodeStatus:
            ids = by_status.get(status, [])
            if ids:
                lines.append(f"  {status.value:<12}: {len(ids)} signal(s) -- {ids}")
        return "\n".join(lines) if lines else "  (empty -- no evidence accumulated yet)"


# =============================================================================
# 4. PORTED subset: NodeKind, SelfGraphNode
#    (copied from core/self_model/self_graph.py -- Kind axis only, the
#    full edge-typed SelfGraph isn't needed for this validation run)
# =============================================================================

class NodeKind(Enum):
    OBSERVATION = "observation"
    MECHANISM = "mechanism"
    HANDLE = "handle"
    ATOMIC_SIGNAL = "atomic_signal"
    COMPOSITE_SIGNAL = "composite_signal"
    DECOY = "decoy"
    UNKNOWN = "unknown"


@dataclass
class SelfGraphNode:
    node_id: str
    kind: NodeKind = NodeKind.UNKNOWN
    kind_confidence: float = 0.0
    evidence: dict = field(default_factory=dict)


# =============================================================================
# 5. NEW: KnowledgeUnitView
#    (docs/knowledge_unit_reconciliation.md, Option B -- thin, stateless,
#    read-only wrapper. Holds no state of its own.)
# =============================================================================

class KnowledgeUnitView:
    def __init__(self, kind_node: SelfGraphNode, world_model: WorldModel):
        self._kind_node = kind_node
        self._wm = world_model

    @property
    def node_id(self) -> str:
        return self._kind_node.node_id

    @property
    def kind(self) -> NodeKind:
        return self._kind_node.kind

    @property
    def kind_confidence(self) -> float:
        return self._kind_node.kind_confidence

    @property
    def status(self) -> Optional[NodeStatus]:
        node = self._wm.nodes.get(self.node_id)
        return node.status if node else None

    @property
    def endogeneity_p_value(self) -> Optional[float]:
        node = self._wm.nodes.get(self.node_id)
        return node.endogeneity_p_value if node else None

    @property
    def flip_rate(self) -> Optional[float]:
        node = self._wm.nodes.get(self.node_id)
        return node.flip_rate if node else None

    def __repr__(self):
        return (f"KnowledgeUnitView({self.node_id!r}, kind={self.kind.value}, "
                f"kind_confidence={self.kind_confidence:.2f}, "
                f"status={self.status.value if self.status else None}, "
                f"endogeneity_p={self.endogeneity_p_value}, "
                f"flip_rate={self.flip_rate})")


# =============================================================================
# 6. NEW: EvidenceManager
#    (docs/knowledge_unit_reconciliation.md, Option A+C -- hard-required
#    intervention arg, two separate gates)
# =============================================================================

POLICY_RELEVANCE_FLIP_THRESHOLD = 0.3  # PROVISIONAL, see module docstring §3


class EvidenceManager:
    def __init__(self, world_model: WorldModel, self_graph_nodes: Dict[str, SelfGraphNode]):
        self.wm = world_model
        self.sg_nodes = self_graph_nodes

    def _ensure_node(self, node_id: str) -> SelfGraphNode:
        if node_id not in self.sg_nodes:
            self.sg_nodes[node_id] = SelfGraphNode(node_id=node_id)
        return self.sg_nodes[node_id]

    def record_endogeneity(self, node_id: str, p_value: float, significant: bool) -> None:
        """Correlational gate only. If significant, node becomes a
        WorldModel CANDIDATE. If not, no WorldModel node is created --
        Question Generator has nothing further to ask (§3.2 of the
        design doc), kind stays UNKNOWN (we asked one question, got a
        no, and did not investigate further -- NOT the same as
        confidently classifying it DECOY, which would need its own
        evidence)."""
        node = self._ensure_node(node_id)
        node.evidence["endogeneity_p_value"] = p_value
        node.evidence["endogeneity_significant"] = significant
        if significant:
            self.wm.add_candidate(node_id, p_value)

    def record_intervention(self, node_id: str, handle_id: str, flip_rate: float,
                             evidence: Optional[dict] = None) -> None:
        """intervention_evidence (handle_id, flip_rate) is NOT optional
        -- there is no code path to ACCEPTED that skips this method
        with real values. Threshold is provisional, see module docstring."""
        node = self._ensure_node(node_id)
        if flip_rate >= POLICY_RELEVANCE_FLIP_THRESHOLD:
            self.wm.accept(node_id, handle_id, flip_rate, evidence)
            node.kind = NodeKind.MECHANISM
            node.kind_confidence = min(1.0, flip_rate)
        else:
            self.wm.reject(node_id, handle_id, flip_rate)
            node.kind = NodeKind.DECOY
            node.kind_confidence = 0.5


# =============================================================================
# 7. NEW: QuestionGenerator
#    (docs/knowledge_unit_reconciliation.md Option B -- independent
#    component, NOT a Hypothesis Generator replacement. Narrow scope
#    here: exactly the two example questions from the reconciliation
#    note, exercised on hand-picked candidates only -- see design doc §5
#    for why open-ended use is out of scope for Phase 2.)
# =============================================================================

class QuestionGenerator:
    def next_question(self, kuv: KnowledgeUnitView) -> Optional[str]:
        if kuv.status is None:
            return "is_internal"          # -> Endogeneity test
        if kuv.status == NodeStatus.CANDIDATE:
            return "is_controllable"      # -> Policy-Relevance test
        return None                        # ACCEPTED/REJECTED: nothing further


# =============================================================================
# 8. NEW: RunningTotalEnvironment
#    Simplification #1 (see module docstring): uses Phase 1's VALIDATED
#    "{T} + {n} =" format, looped, not the design doc's illustrative
#    "Total: {T}. Add {n}. New total:" phrasing.
# =============================================================================

class RunningTotalEnvironment:
    def __init__(self, model, operand_range=(1, 4), start_total=0, seed=0):
        self.model = model
        self.operand_range = operand_range
        self.start_total = start_total
        self.rng = np.random.default_rng(seed)
        self.T = start_total
        self.n_anomalies = 0
        self.n_steps_run = 0

    def reset(self):
        self.T = self.start_total
        self.n_anomalies = 0
        self.n_steps_run = 0

    def step(self) -> dict:
        n = int(self.rng.integers(self.operand_range[0], self.operand_range[1] + 1))
        prompt = f"{self.T} + {n} ="
        tokens = self.model.to_tokens(prompt)
        logits = self.model(tokens)
        next_token_id = int(logits[0, -1, :].argmax().item())
        next_token_str = self.model.to_string(torch.tensor([next_token_id]))

        try:
            T_next = int(next_token_str.strip())
            valid = 0 <= T_next <= 9
        except ValueError:
            valid = False
            T_next = None

        if not valid:
            self.n_anomalies += 1
            # Fall back to wrapping the TRUE sum mod 10 so the loop can
            # keep running -- this step's action is still whatever the
            # model generated (for the action encoding / endogeneity
            # test below), only the CONTINUATION uses the fallback.
            T_next_for_continuation = (self.T + n) % 10
        else:
            T_next_for_continuation = T_next

        result = {
            "T": self.T, "n": n, "prompt": prompt,
            "generated_token_id": next_token_id,
            "generated_token_str": next_token_str,
            "T_next_raw": T_next,          # None if unparseable
            "T_next_valid": valid,
            "T_next": T_next_for_continuation,  # always 0-9, used to continue the loop
        }
        self.T = T_next_for_continuation
        self.n_steps_run += 1
        return result


def action_onehot_from_digit(digit: int, n_categories: int = 10) -> list:
    """Simplification #2 (see module docstring): plain one-hot, not the
    k-1 reduced encoding evaluation/action_encoding.py uses."""
    row = [0.0] * n_categories
    row[digit] = 1.0
    return row


# =============================================================================
# 9. NEW: candidate signal extraction -- fixed, out-of-sample-calibrated
#    probe directions (calibrated ONCE, before any trajectory, on an
#    independent set, per Phase 1's corrected probing methodology)
# =============================================================================

def calibrated_addition_pairs(operand_max=9, seed=7):
    """Same balanced-enumeration, single-token-answer-filtered approach
    Phase 1's probing fix used -- avoids the CV-leakage failure mode
    documented in docs/llm_phase1_addition_circuit.md."""
    rng = np.random.default_rng(seed)
    candidates = [(a, b) for a in range(0, 10) for b in range(0, 10) if a + b <= operand_max]
    valid = []
    for a, b in candidates:
        ans_ids = model.to_tokens(f" {a + b}", prepend_bos=False)
        if ans_ids.shape[-1] == 1:
            valid.append((a, b))
    rng.shuffle(valid)
    return valid


def calibrate_direction(layers: List[int], calibration_pairs, position: int = -1) -> Dict[int, np.ndarray]:
    """Fit one Ridge probe direction per layer on an INDEPENDENT
    calibration set (not reused for trajectory evaluation later --
    avoids circularity). Returns {layer: normalized weight vector}."""
    acts_by_layer = {layer: [] for layer in layers}
    sums = []
    for a, b in calibration_pairs:
        prompt = f"{a} + {b} ="
        tokens = model.to_tokens(prompt)
        _, cache = model.run_with_cache(tokens)
        for layer in layers:
            acts_by_layer[layer].append(cache["resid_post", layer][0, position, :].cpu().numpy())
        sums.append(a + b)
    sums = np.array(sums)

    directions = {}
    for layer in layers:
        X = np.stack(acts_by_layer[layer])
        ridge = Ridge(alpha=10.0)
        ridge.fit(X, sums)
        w = ridge.coef_
        w = w / (np.linalg.norm(w) + 1e-8)
        directions[layer] = w
    return directions


def project_resid(prompt: str, layer: int, direction: np.ndarray, position: int = -1) -> float:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens)
    act = cache["resid_post", layer][0, position, :].cpu().numpy()
    return float(np.dot(act, direction))


def cluster_signal_fn(directions_9_11: Dict[int, np.ndarray]):
    def fn(step_result: dict) -> float:
        vals = [project_resid(step_result["prompt"], layer, directions_9_11[layer])
                for layer in (9, 10, 11)]
        return float(np.mean(vals))
    return fn


def negctrl_signal_fn(directions_0_3: Dict[int, np.ndarray]):
    def fn(step_result: dict) -> float:
        vals = [project_resid(step_result["prompt"], layer, directions_0_3[layer])
                for layer in (0, 1, 2, 3)]
        return float(np.mean(vals))
    return fn


# =============================================================================
# 10. NEW: Policy-Relevance flip-rate via activation patching
#     Ports the FLIP-RATE METRIC from core/intervention/policy_relevance.py,
#     not its body (which is env/registry/planner-specific, see design doc §4).
# =============================================================================

def policy_relevance_flip_rate(env: RunningTotalEnvironment, layers: List[int],
                                n_trials: int = 30, position: int = -1, seed: int = 1) -> float:
    """For n_trials random steps: run a 'clean' prompt, a 'donor' prompt
    (same T, different n), patch the donor's activation at `layers`
    (resid_post, given position) into the clean run, and check whether
    the model's generated next-token ACTION changes. flip_rate = fraction
    of trials where it does."""
    rng = np.random.default_rng(seed)
    flips = 0
    for _ in range(n_trials):
        T = int(rng.integers(0, 10))
        n_clean = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        while n_donor == n_clean:
            n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))

        clean_prompt = f"{T} + {n_clean} ="
        donor_prompt = f"{T} + {n_donor} ="

        clean_tokens = model.to_tokens(clean_prompt)
        donor_tokens = model.to_tokens(donor_prompt)
        if clean_tokens.shape[-1] != donor_tokens.shape[-1]:
            continue  # alignment guard, same discipline as Phase 1

        clean_logits_orig = model(clean_tokens)
        original_action = int(clean_logits_orig[0, -1, :].argmax().item())

        _, donor_cache = model.run_with_cache(donor_tokens)

        def patch_hook(activation, hook, donor_cache=donor_cache, position=position):
            activation[:, position, :] = donor_cache[hook.name][:, position, :]
            return activation

        hooks = [(f"blocks.{layer}.hook_resid_post", patch_hook) for layer in layers]
        patched_logits = model.run_with_hooks(clean_tokens, fwd_hooks=hooks)
        patched_action = int(patched_logits[0, -1, :].argmax().item())

        if patched_action != original_action:
            flips += 1
    return flips / n_trials if n_trials > 0 else 0.0


def policy_relevance_flip_rate_head(env: RunningTotalEnvironment, layer: int, head: int,
                                     n_trials: int = 30, position: int = -1, seed: int = 1) -> float:
    """Same as above but patches a single attention head's OUTPUT
    (hook_z, this head's slice) rather than the whole layer's resid_post."""
    rng = np.random.default_rng(seed)
    flips = 0
    for _ in range(n_trials):
        T = int(rng.integers(0, 10))
        n_clean = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        while n_donor == n_clean:
            n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))

        clean_prompt = f"{T} + {n_clean} ="
        donor_prompt = f"{T} + {n_donor} ="
        clean_tokens = model.to_tokens(clean_prompt)
        donor_tokens = model.to_tokens(donor_prompt)
        if clean_tokens.shape[-1] != donor_tokens.shape[-1]:
            continue

        clean_logits_orig = model(clean_tokens)
        original_action = int(clean_logits_orig[0, -1, :].argmax().item())

        _, donor_cache = model.run_with_cache(donor_tokens)

        def patch_head_hook(activation, hook, donor_cache=donor_cache, position=position, head=head):
            activation[:, position, head, :] = donor_cache[hook.name][:, position, head, :]
            return activation

        hook_name = f"blocks.{layer}.attn.hook_z"
        patched_logits = model.run_with_hooks(clean_tokens, fwd_hooks=[(hook_name, patch_head_hook)])
        patched_action = int(patched_logits[0, -1, :].argmax().item())

        if patched_action != original_action:
            flips += 1
    return flips / n_trials if n_trials > 0 else 0.0


# =============================================================================
# 11. Main pipeline: candidate -> KnowledgeUnitView -> QuestionGenerator
#     -> (Endogeneity via HypothesisEngine) -> EvidenceManager
#     -> (Policy-Relevance via patching) -> EvidenceManager
# =============================================================================

def run_candidate_validation(node_id: str, signal_fn, other_signal_fns: Dict[str, callable],
                              flip_rate_fn, world_model: WorldModel,
                              self_graph_nodes: Dict[str, SelfGraphNode],
                              evidence_mgr: EvidenceManager, question_gen: QuestionGenerator,
                              n_steps: int = 100, n_flip_trials: int = 30,
                              env_seed: int = 0) -> KnowledgeUnitView:

    print(f"\n=== Candidate: {node_id} ===")
    env = RunningTotalEnvironment(model, seed=env_seed)
    engine = HypothesisEngine(signal_ids=[node_id] + list(other_signal_fns.keys()))

    kuv = KnowledgeUnitView(self_graph_nodes.setdefault(node_id, SelfGraphNode(node_id)), world_model)
    q = question_gen.next_question(kuv)
    assert q == "is_internal", f"expected first question to be is_internal, got {q}"

    print(f"[{node_id}] collecting {n_steps}-step trajectory for Endogeneity test...")
    env.reset()
    for _ in range(n_steps):
        step_result = env.step()
        signal_values = {node_id: signal_fn(step_result)}
        for oid, fn in other_signal_fns.items():
            signal_values[oid] = fn(step_result)
        action_row = action_onehot_from_digit(step_result["T_next"])
        engine.record_step(signal_values, action_row)

    print(f"[{node_id}] generation anomalies (unparseable next-total): "
          f"{env.n_anomalies}/{env.n_steps_run}")

    endog_result = engine.test_endogeneity(min_samples=60)
    if node_id not in endog_result:
        print(f"[{node_id}] WARNING: not enough samples for Endogeneity test "
              f"({engine.n_samples()} collected, need >=60) -- increase n_steps")
        return kuv

    p_info = endog_result[node_id]
    print(f"[{node_id}] Endogeneity: p={p_info['p_value']:.4f}, "
          f"significant={p_info['significant']}")
    evidence_mgr.record_endogeneity(node_id, p_info["p_value"], p_info["significant"])

    kuv = KnowledgeUnitView(self_graph_nodes[node_id], world_model)
    q = question_gen.next_question(kuv)
    if q != "is_controllable":
        print(f"[{node_id}] Question Generator: no further question "
              f"(status={kuv.status}) -- stopping here.")
        return kuv

    print(f"[{node_id}] running Policy-Relevance flip-rate test "
          f"({n_flip_trials} trials)...")
    flip_rate = flip_rate_fn(env, n_flip_trials)
    handle_id = f"donor_activation@{node_id}"
    print(f"[{node_id}] Policy-Relevance: flip_rate={flip_rate:.3f} "
          f"(threshold={POLICY_RELEVANCE_FLIP_THRESHOLD})")
    evidence_mgr.record_intervention(node_id, handle_id, flip_rate)

    kuv = KnowledgeUnitView(self_graph_nodes[node_id], world_model)
    print(f"[{node_id}] final: {kuv}")
    return kuv


if __name__ == "__main__":
    calibration_pairs = calibrated_addition_pairs()
    print(f"[calibration] {len(calibration_pairs)} unique (a,b) pairs for probe fitting")

    directions_9_11 = calibrate_direction([9, 10, 11], calibration_pairs)
    directions_0_3 = calibrate_direction([0, 1, 2, 3], calibration_pairs)

    world_model = WorldModel()
    self_graph_nodes: Dict[str, SelfGraphNode] = {}
    evidence_mgr = EvidenceManager(world_model, self_graph_nodes)
    question_gen = QuestionGenerator()

    cluster_fn = cluster_signal_fn(directions_9_11)
    negctrl_fn = negctrl_signal_fn(directions_0_3)

    # ---- Positive check: layer 9-11 cluster (primary target, stable per
    # Phase 1's bootstrap) vs. the early-layer signal as its "other" ----
    kuv_cluster = run_candidate_validation(
        node_id="cluster_layers_9_11",
        signal_fn=cluster_fn,
        other_signal_fns={"early_layers_0_3": negctrl_fn},
        flip_rate_fn=lambda env, n: policy_relevance_flip_rate(env, [9, 10, 11], n_trials=n),
        world_model=world_model, self_graph_nodes=self_graph_nodes,
        evidence_mgr=evidence_mgr, question_gen=question_gen,
    )

    # ---- Positive check: head (9,1) specifically -- KNOWN to be
    # bootstrap-unstable (48.9% argmax win rate, Phase 1). A failure here
    # is inconclusive between "port bug" and "unstable ground truth" --
    # see docs/phase2_design_consolidated.md §4.2. ----
    kuv_head = run_candidate_validation(
        node_id="head_9_1",
        signal_fn=lambda step_result: project_resid(step_result["prompt"], 9, directions_9_11[9]),
        other_signal_fns={"early_layers_0_3": negctrl_fn},
        flip_rate_fn=lambda env, n: policy_relevance_flip_rate_head(env, layer=9, head=1, n_trials=n),
        world_model=world_model, self_graph_nodes=self_graph_nodes,
        evidence_mgr=evidence_mgr, question_gen=question_gen,
    )

    # ---- Negative check: early layers 0-3 (expected to fail
    # Policy-Relevance even if Endogeneity happens to pass) ----
    kuv_negctrl = run_candidate_validation(
        node_id="negctrl_layers_0_3",
        signal_fn=negctrl_fn,
        other_signal_fns={"cluster_layers_9_11": cluster_fn},
        flip_rate_fn=lambda env, n: policy_relevance_flip_rate(env, [0, 1, 2, 3], n_trials=n),
        world_model=world_model, self_graph_nodes=self_graph_nodes,
        evidence_mgr=evidence_mgr, question_gen=question_gen,
    )

    print("\n" + "=" * 60)
    print("WorldModel summary:")
    print(world_model.self_model_summary())
    print("\nExpected (if the port is working):")
    print("  cluster_layers_9_11 -> ACCEPTED (endogenous + policy-relevant)")
    print("  head_9_1            -> ACCEPTED or inconclusive (bootstrap-unstable, see caveat)")
    print("  negctrl_layers_0_3  -> CANDIDATE-but-REJECTED, or never a candidate at all")

    np.savez(
        "phase2_validation_results.npz",
        cluster_status=str(kuv_cluster.status),
        cluster_flip_rate=kuv_cluster.flip_rate,
        head_status=str(kuv_head.status),
        head_flip_rate=kuv_head.flip_rate,
        negctrl_status=str(kuv_negctrl.status),
        negctrl_flip_rate=kuv_negctrl.flip_rate,
    )
    print("\nSaved phase2_validation_results.npz")

/tmp/ipykernel_943/1582648862.py:58: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
[calibration] 55 unique (a,b) pairs for probe fitting

=== Candidate: cluster_layers_9_11 ===
[cluster_layers_9_11] collecting 100-step trajectory for Endogeneity test...
[cluster_layers_9_11] generation anomalies (unparseable next-total): 0/100
[cluster_layers_9_11] Endogeneity: p=1.0000, significant=False
[cluster_layers_9_11] Question Generator: no further question (status=None) -- stopping here.

=== Candidate: head_9_1 ===
[head_9_1] collecting 100-step trajectory for Endogeneity test...
[head_9_1] generation anomalies (unparseable next-total): 0/100
[head_9_1] Endogeneity: p=1.0000, significant=False
[head_9_1] Question Generator: no further question (status=None) -- stopping here.

=== Candidate: negctrl_layers_0_3 ===
[negctrl_layers_0_3] collecting 100-step trajectory for Endogeneity test...
[negctrl_layers_0_3] generation anomalies (unparseable next-total): 0/100
[negctrl_layers_0_3] Endogeneity: p=1.0000, significant=False


In [3]:
"""
Diagnostic follow-up: why did all three candidates get p=1.0000 exactly?

Run this in the SAME Colab session as phase2_running_total_colab.py --
reuses `model`, and redefines a small instrumented copy of
conditional_granger_f_test that prints the intermediate numbers instead
of just returning (f_stat, p_value). This is diagnostic-only; it does
not change the actual test used by HypothesisEngine.
"""

import numpy as np

# ---------------------------------------------------------------------------
# 1. Collect ONE candidate's trajectory data the same way the main script
#    does, so we have real numbers to inspect (reuses functions already
#    defined in this session from phase2_running_total_colab.py).
# ---------------------------------------------------------------------------

directions_9_11 = calibrate_direction([9, 10, 11], calibrated_addition_pairs())
directions_0_3 = calibrate_direction([0, 1, 2, 3], calibrated_addition_pairs())
cluster_fn = cluster_signal_fn(directions_9_11)
negctrl_fn = negctrl_signal_fn(directions_0_3)

env = RunningTotalEnvironment(model, seed=0)
env.reset()

signal_rows = []
action_rows = []
T_values = []
for _ in range(100):
    step_result = env.step()
    signal_rows.append([cluster_fn(step_result), negctrl_fn(step_result)])
    action_rows.append(action_onehot_from_digit(step_result["T_next"]))
    T_values.append(step_result["T_next"])

matrix = np.array(signal_rows)       # [100, 2] -- cluster, negctrl
action_onehot = np.array(action_rows)  # [100, 10]
T_values = np.array(T_values)

# ---------------------------------------------------------------------------
# 2. Basic sanity: is the candidate signal actually varying? Is T actually
#    exploring a range of values (Excitation, Principle 8)?
# ---------------------------------------------------------------------------

print("=== Basic variance / excitation checks ===")
print(f"cluster signal:  mean={matrix[:,0].mean():.4f}, std={matrix[:,0].std():.4f}, "
      f"min={matrix[:,0].min():.4f}, max={matrix[:,0].max():.4f}")
print(f"negctrl signal:  mean={matrix[:,1].mean():.4f}, std={matrix[:,1].std():.4f}, "
      f"min={matrix[:,1].min():.4f}, max={matrix[:,1].max():.4f}")
print(f"T_values: unique={sorted(set(T_values.tolist()))}, "
      f"counts={np.bincount(T_values, minlength=10)}")
print(f"action_onehot: rank={np.linalg.matrix_rank(action_onehot)} "
      f"(out of {action_onehot.shape[1]} columns) -- if rank < n_categories seen, "
      f"some digits never occurred (dead columns)")

# ---------------------------------------------------------------------------
# 3. Instrumented copy of conditional_granger_f_test -- same math, but
#    prints rss_r, rss_u, df1, df2, and matrix ranks/condition numbers.
# ---------------------------------------------------------------------------

def diagnose_granger(target_col, other_cols, action_onehot_arr, lag=3, label=""):
    T = target_col.shape[0]
    y = target_col[lag:]
    n = y.shape[0]

    self_lags = build_lagged(target_col.reshape(-1, 1), lag)
    other_lags = build_lagged(other_cols, lag) if other_cols.shape[1] > 0 else np.zeros((n, 0))

    action_contemporaneous = action_onehot_arr[lag:]
    action_past_lags = build_lagged(action_onehot_arr, lag)
    action_history = np.concatenate([action_contemporaneous, action_past_lags], axis=1)

    restricted_X = np.concatenate([self_lags, other_lags], axis=1)
    unrestricted_X = np.concatenate([restricted_X, action_history], axis=1)

    k_r = restricted_X.shape[1] + 1
    k_u = unrestricted_X.shape[1] + 1
    q = k_u - k_r

    rss_r = ols_rss(y, restricted_X)
    rss_u = ols_rss(y, unrestricted_X)

    df1 = q
    df2 = n - k_u

    restricted_design = np.concatenate([np.ones((restricted_X.shape[0], 1)), restricted_X], axis=1)
    unrestricted_design = np.concatenate([np.ones((unrestricted_X.shape[0], 1)), unrestricted_X], axis=1)

    print(f"\n--- {label} ---")
    print(f"n (usable samples) = {n}")
    print(f"restricted_X shape = {restricted_X.shape}, k_r = {k_r}, "
          f"rank = {np.linalg.matrix_rank(restricted_design)} (design incl. intercept)")
    print(f"unrestricted_X shape = {unrestricted_X.shape}, k_u = {k_u}, "
          f"rank = {np.linalg.matrix_rank(unrestricted_design)} (design incl. intercept)")
    print(f"df1 = q = {q}, df2 = n - k_u = {df2}")
    print(f"rss_r = {rss_r:.6e}, rss_u = {rss_u:.6e}, "
          f"rss_r - rss_u = {rss_r - rss_u:.6e}")
    print(f"condition number of unrestricted design: "
          f"{np.linalg.cond(unrestricted_design):.3e}")

    if df2 <= 0 or rss_u <= 0:
        print("!! df2<=0 or rss_u<=0 -- test returns p=1.0 by construction, this IS the bug path")
        return
    f_stat_raw = ((rss_r - rss_u) / df1) / (rss_u / df2)
    print(f"f_stat (before clipping to >=0) = {f_stat_raw:.6f}")


diagnose_granger(matrix[:, 0], matrix[:, 1:2], action_onehot, lag=3, label="cluster_layers_9_11")
diagnose_granger(matrix[:, 1], matrix[:, 0:1], action_onehot, lag=3, label="negctrl_layers_0_3")

=== Basic variance / excitation checks ===
cluster signal:  mean=-4.9178, std=2.0899, min=-8.1356, max=-2.0822
negctrl signal:  mean=-4.9403, std=0.4405, min=-5.5870, max=-4.3454
T_values: unique=[0], counts=[100   0   0   0   0   0   0   0   0   0]
action_onehot: rank=1 (out of 10 columns) -- if rank < n_categories seen, some digits never occurred (dead columns)

--- cluster_layers_9_11 ---
n (usable samples) = 97
restricted_X shape = (97, 6), k_r = 7, rank = 7 (design incl. intercept)
unrestricted_X shape = (97, 46), k_u = 47, rank = 7 (design incl. intercept)
df1 = q = 40, df2 = n - k_u = 50
rss_r = 3.948919e+02, rss_u = 3.948919e+02, rss_r - rss_u = -1.136868e-13
condition number of unrestricted design: inf
f_stat (before clipping to >=0) = -0.000000

--- negctrl_layers_0_3 ---
n (usable samples) = 97
restricted_X shape = (97, 6), k_r = 7, rank = 7 (design incl. intercept)
unrestricted_X shape = (97, 46), k_u = 47, rank = 7 (design incl. intercept)
df1 = q = 40, df2 = n - k_u = 50


In [4]:
"""
Fix for the T=0 absorbing-trap bug found via phase2_diagnose_granger.py.

Two changes from the original RunningTotalEnvironment:
  1. start_total defaults to a draw from {1,2,3,4} -- Phase 1's actually-
     validated operand range -- instead of 0, which was never tested in
     Phase 1 and appears to be an absorbing fixed point for GPT-2's
     greedy completion of "0 + n =".
  2. A "stuck" watchdog: if T doesn't change for N_STUCK_WARN consecutive
     steps, print a warning immediately rather than silently completing
     a whole degenerate trajectory (which is what happened last time --
     100 steps, zero warning, and a completely uninformative dataset as
     a result).

IMPORTANT: this does NOT forcibly clip or wrap the model's VALID
generated outputs into a fixed range. Endogeneity requires T_{t+1} to be
literally whatever the model generated -- clipping a valid output would
mean testing a constructed signal, not the model's actual behavior. If
the trajectory wanders into T=0 (or any other value) LATER through
genuine generation and gets stuck there too, that's real data about the
model, not a bug -- the watchdog will flag it either way so it's visible
rather than silently consumed.

Run this in the SAME Colab session (reuses `model`). Prints the first 15
(T, n, generated, T_next) steps directly, so we can SEE the trajectory
move before trusting statistics computed on top of it.
"""

import numpy as np
import torch


class RunningTotalEnvironmentV2:
    N_STUCK_WARN = 8  # consecutive identical T values before warning

    def __init__(self, model, operand_range=(1, 4), start_range=(1, 4), seed=0):
        self.model = model
        self.operand_range = operand_range
        self.start_range = start_range
        self.rng = np.random.default_rng(seed)
        self.T = None
        self.n_anomalies = 0
        self.n_steps_run = 0
        self._stuck_count = 0
        self._last_T = None
        self._warned_stuck = False

    def reset(self):
        self.T = int(self.rng.integers(self.start_range[0], self.start_range[1] + 1))
        self.n_anomalies = 0
        self.n_steps_run = 0
        self._stuck_count = 0
        self._last_T = None
        self._warned_stuck = False

    def step(self, verbose=False) -> dict:
        n = int(self.rng.integers(self.operand_range[0], self.operand_range[1] + 1))
        prompt = f"{self.T} + {n} ="
        tokens = self.model.to_tokens(prompt)
        logits = self.model(tokens)
        next_token_id = int(logits[0, -1, :].argmax().item())
        next_token_str = self.model.to_string(torch.tensor([next_token_id]))

        try:
            T_next = int(next_token_str.strip())
            valid = 0 <= T_next <= 9
        except ValueError:
            valid = False
            T_next = None

        if not valid:
            self.n_anomalies += 1
            T_next_for_continuation = (self.T + n) % 10
        else:
            T_next_for_continuation = T_next

        if verbose:
            print(f"  step {self.n_steps_run}: T={self.T}, n={n}, "
                  f"prompt={prompt!r}, generated={next_token_str!r}, "
                  f"valid={valid}, T_next={T_next_for_continuation}")

        # stuck watchdog
        if self._last_T is not None and T_next_for_continuation == self._last_T:
            self._stuck_count += 1
        else:
            self._stuck_count = 0
        self._last_T = T_next_for_continuation
        if self._stuck_count >= self.N_STUCK_WARN and not self._warned_stuck:
            print(f"  !! WARNING: T has been stuck at {T_next_for_continuation} for "
                  f"{self._stuck_count + 1} consecutive steps -- trajectory may be "
                  f"in an absorbing state, same failure mode as the T=0 bug")
            self._warned_stuck = True

        result = {
            "T": self.T, "n": n, "prompt": prompt,
            "generated_token_id": next_token_id,
            "generated_token_str": next_token_str,
            "T_next_raw": T_next,
            "T_next_valid": valid,
            "T_next": T_next_for_continuation,
        }
        self.T = T_next_for_continuation
        self.n_steps_run += 1
        return result


# ---------------------------------------------------------------------------
# Quick direct check: print the first 15 steps for a few different seeds,
# to SEE whether the loop moves now, before re-running the full pipeline.
# ---------------------------------------------------------------------------

for seed in [0, 1, 2]:
    print(f"\n=== seed={seed} ===")
    env = RunningTotalEnvironmentV2(model, seed=seed)
    env.reset()
    print(f"start_total = {env.T}")
    for _ in range(15):
        env.step(verbose=True)


=== seed=0 ===
start_total = 4
  step 0: T=4, n=3, prompt='4 + 3 =', generated=' 2', valid=True, T_next=2
  step 1: T=2, n=3, prompt='2 + 3 =', generated=' 4', valid=True, T_next=4
  step 2: T=4, n=2, prompt='4 + 2 =', generated=' 2', valid=True, T_next=2
  step 3: T=2, n=2, prompt='2 + 2 =', generated=' 2', valid=True, T_next=2
  step 4: T=2, n=1, prompt='2 + 1 =', generated=' 1', valid=True, T_next=1
  step 5: T=1, n=1, prompt='1 + 1 =', generated=' 1', valid=True, T_next=1
  step 6: T=1, n=1, prompt='1 + 1 =', generated=' 1', valid=True, T_next=1
  step 7: T=1, n=1, prompt='1 + 1 =', generated=' 1', valid=True, T_next=1
  step 8: T=1, n=4, prompt='1 + 4 =', generated=' 1', valid=True, T_next=1
  step 9: T=1, n=3, prompt='1 + 3 =', generated=' 4', valid=True, T_next=4
  step 10: T=4, n=4, prompt='4 + 4 =', generated=' 4', valid=True, T_next=4
  step 11: T=4, n=3, prompt='4 + 3 =', generated=' 2', valid=True, T_next=2
  step 12: T=2, n=3, prompt='2 + 3 =', generated=' 4', valid=True,

In [5]:
"""
Exact baseline accuracy: GPT-2 small's greedy (argmax) completion for
every one of the 16 (a,b) combos in [1,4]x[1,4] -- the exact domain
Phase 1's patching validated and Phase 2's running-total task uses.

Deterministic -- greedy decoding on a fixed prompt always gives the same
answer, so no repeated trials needed, just all 16 combos once each.

Run in the SAME Colab session (reuses `model`).
"""

import torch

results = []
for a in range(1, 5):
    for b in range(1, 5):
        prompt = f"{a} + {b} ="
        tokens = model.to_tokens(prompt)
        logits = model(tokens)
        pred_id = int(logits[0, -1, :].argmax().item())
        pred_str = model.to_string(torch.tensor([pred_id])).strip()
        correct_answer = str(a + b)
        is_correct = (pred_str == correct_answer)

        # also record the RANK of the correct answer's logit, and its
        # probability, not just whether it happened to win argmax --
        # distinguishes "close but token-frequency lost the tiebreak"
        # from "correct answer wasn't even a serious contender"
        correct_token_ids = model.to_tokens(f" {a+b}", prepend_bos=False)
        correct_id = int(correct_token_ids[0, 0].item())
        probs = torch.softmax(logits[0, -1, :], dim=-1)
        correct_prob = float(probs[correct_id].item())
        correct_rank = int((logits[0, -1, :] > logits[0, -1, correct_id]).sum().item()) + 1

        results.append({
            "a": a, "b": b, "correct": a + b,
            "predicted": pred_str, "is_correct": is_correct,
            "correct_answer_prob": correct_prob,
            "correct_answer_rank": correct_rank,
        })

n_correct = sum(r["is_correct"] for r in results)
print(f"=== Exact greedy accuracy on all 16 pairs in [1,4]x[1,4] ===")
print(f"{n_correct}/16 correct ({100*n_correct/16:.1f}%)\n")

print(f"{'a+b':>5} | {'correct':>7} | {'predicted':>9} | {'✓/✗':>3} | "
      f"{'P(correct)':>10} | {'rank(correct)':>13}")
print("-" * 65)
for r in sorted(results, key=lambda r: (r['a'], r['b'])):
    mark = "✓" if r["is_correct"] else "✗"
    print(f"{r['a']}+{r['b']:<3} | {r['correct']:>7} | {r['predicted']:>9} | "
          f"{mark:>3} | {r['correct_answer_prob']:>10.4f} | {r['correct_answer_rank']:>13}")

print(f"\nMean P(correct answer), regardless of whether it won argmax: "
      f"{sum(r['correct_answer_prob'] for r in results)/16:.4f}")
print(f"Mean rank of correct answer (1 = top): "
      f"{sum(r['correct_answer_rank'] for r in results)/16:.2f}")
print(f"How many pairs have correct answer in top-3 by probability: "
      f"{sum(1 for r in results if r['correct_answer_rank'] <= 3)}/16")

=== Exact greedy accuracy on all 16 pairs in [1,4]x[1,4] ===
2/16 correct (12.5%)

  a+b | correct | predicted | ✓/✗ | P(correct) | rank(correct)
-----------------------------------------------------------------
1+1   |       2 |         1 |   ✗ |     0.1357 |             2
1+2   |       3 |         3 |   ✓ |     0.1914 |             1
1+3   |       4 |         4 |   ✓ |     0.0978 |             1
1+4   |       5 |         1 |   ✗ |     0.0525 |             6
2+1   |       3 |         1 |   ✗ |     0.0652 |             3
2+2   |       4 |         2 |   ✗ |     0.0763 |             4
2+3   |       5 |         4 |   ✗ |     0.0518 |             5
2+4   |       6 |         4 |   ✗ |     0.0590 |             5
3+1   |       4 |         1 |   ✗ |     0.0566 |             4
3+2   |       5 |         2 |   ✗ |     0.0440 |             5
3+3   |       6 |         3 |   ✗ |     0.0467 |             6
3+4   |       7 |         4 |   ✗ |     0.0302 |             8
4+1   |       5 |         1 |   